# Dependencies and Imports

In [ ]:
!pip install scanpy cellxgene_census

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 104.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
import json
import random
import cellxgene_census
import re
import os
import gc
import scipy
import anndata

In [ ]:
# Connect to CELLxGENE API
census = cellxgene_census.open_soma()

The "stable" release is currently 2025-01-30. Specify 'census_version="2025-01-30"' in future calls to open_soma() to ensure data consistency.
INFO:cellxgene_census:The "stable" release is currently 2025-01-30. Specify 'census_version="2025-01-30"' in future calls to open_soma() to ensure data consistency.


# Inspect CELLxGENE Database

In [ ]:
#disease
disease_cell_metadata = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["disease"])

print(disease_cell_metadata.drop_duplicates())


#sex
sex_cell_metadata = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["sex"])

print(sex_cell_metadata.drop_duplicates())

#celltype
celltype_cell_metadata = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["cell_type"])

print(celltype_cell_metadata.drop_duplicates())

#tissue
tissue_cell_metadata = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["tissue"])

print(tissue_cell_metadata.drop_duplicates())

#tissue_general
tissue_general_cell_metadata = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["tissue_general"])

print(tissue_general_cell_metadata.drop_duplicates())

#developmental_stage
developmental_stage_cell_metadata = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["development_stage"])

print(developmental_stage_cell_metadata.drop_duplicates())

                                      disease
0                               breast cancer
565                                    normal
2558                        Alzheimer disease
5739         common variable immunodeficiency
46776                                COVID-19
...                                       ...
92408920                     chronic rhinitis
92409036                pleomorphic carcinoma
92409059            lung large cell carcinoma
92409290         hypersensitivity pneumonitis
92409500  non-specific interstitial pneumonia

[138 rows x 1 columns]
          sex
0      female
650      male
1033  unknown
                                    cell_type
0                            endothelial cell
1                              malignant cell
2                                  fibroblast
4                                  macrophage
8                                    monocyte
...                                       ...
83033607         fibroblast of choroid plexus


In [ ]:
del sex_cell_metadata
del disease_cell_metadata
del celltype_cell_metadata
del tissue_cell_metadata
del tissue_general_cell_metadata
del developmental_stage_cell_metadata

# Sampling from CELLxGENE Database

In [ ]:
def map_dev_stage(stage: str) -> str:
    if not isinstance(stage, str):
        return "unknown"

    stage = stage.lower()

    # Prenatal terms
    if "post-fertilization" in stage or "blastula" in stage or "gastrula" in stage:
        return "prenatal"

    # Try to extract an age and unit (year, month, week)
    match = re.search(r"(\d+)[-\s]*(year|month|week)", stage)
    if match:
        value = int(match.group(1))
        unit = match.group(2)

        if unit == "week":
            return "prenatal"  # still in gestation
        elif unit == "month":
            if value <= 12:
                return "child"
            elif value <= 360:  # 20 years
                return "young_adult"
            elif value <= 720:  # 60 years
                return "adult"
            else:
                return "aged"
        elif unit == "year":
            if value <= 12:
                return "child"
            elif value <= 30:
                return "young_adult"
            elif value <= 60:
                return "adult"
            else:
                return "aged"

    # Fallback: try to infer from known string patterns
    if "child" in stage:
        return "child"
    if "adult" in stage:
        if "young" in stage:
            return "young_adult"
        return "adult"
    if "aged" in stage or "elderly" in stage:
        return "aged"

    return "unknown"


In [ ]:
# Get only the metadata of cell from the database to
obs_df = cellxgene_census.get_obs(census, "homo_sapiens", column_names=["soma_joinid", "development_stage", "disease", "sex", "tissue", "tissue_general", "cell_type", "is_primary_data"])

# Apply binning at development stage
obs_df["dev_stage_group"] = obs_df["development_stage"].apply(map_dev_stage)

# Apply any filtering & sampling logic here
filtered = obs_df[
    #obs_df["sex"].isin(["male", "female"]) &
    #obs_df["dev_stage_group"].ne("unknown") &
    obs_df["tissue_general"].notna() &
    obs_df["tissue_general"].ne("unknown") &
    obs_df["disease"].notna() &
    obs_df["disease"].ne("unknown") &
    obs_df["cell_type"].ne("unknown") &
    obs_df["cell_type"].notna() &
    obs_df["is_primary_data"] == True # removes duplicate entries from the dataset
]


In [ ]:
del obs_df
print(filtered)

           soma_joinid  development_stage   disease     sex  \
5716              5716  26-year-old stage    normal    male   
5717              5717  26-year-old stage    normal    male   
5718              5718  26-year-old stage    normal    male   
5719              5719  26-year-old stage    normal    male   
5720              5720  26-year-old stage    normal    male   
...                ...                ...       ...     ...   
106118162    106118162  88-year-old stage  dementia  female   
106118163    106118163  81-year-old stage  dementia  female   
106118164    106118164  78-year-old stage    normal    male   
106118165    106118165  75-year-old stage    normal    male   
106118166    106118166  81-year-old stage  dementia    male   

                                   tissue tissue_general  \
5716                                blood          blood   
5717                                blood          blood   
5718                                blood          blood   
571

In [ ]:
# Target number of cells
target_total = 100000

# Define our sampling strategy percentages with adjustments
pct_distribution = 0.40
pct_cell_type = 0.25
pct_disease = 0.25
pct_rare = 0.10

# Calculate target counts for each strategy
n_distribution = int(target_total * pct_distribution)
n_cell_type = int(target_total * pct_cell_type)
n_disease = int(target_total * pct_disease)
n_rare = target_total - n_distribution - n_cell_type - n_disease

# Part 1: Distribution-based sampling with tissue adjustment
# Let's check if brain/blood are overrepresented in the original data
tissue_counts = filtered['tissue_general'].value_counts(normalize=True)

# Adjust our sampling to reflect more realistic proportions
tissue_adjustment = {
    'brain': 0.15,  # Cap brain at 15%
    'blood': 0.15   # Cap blood at 15%
}

# Create adjusted weights for distribution sampling
weights = np.ones(len(filtered))

for tissue, cap in tissue_adjustment.items():
    # Calculate how much to downweight these tissues
    current_prop = tissue_counts.get(tissue, 0)
    if current_prop > cap:
        downweight_factor = cap / current_prop
        # Apply downweighting to these tissues
        weights[filtered['tissue_general'] == tissue] = downweight_factor

# Sample with adjusted weights
distribution_sample = filtered.sample(
    n=n_distribution,
    weights=weights,
    random_state=42
)

# Part 2: Cell type representation with improved balance
# Get counts of each cell type
cell_type_counts = filtered['cell_type'].value_counts()

# Calculate the max cells per type (cap at 3% of the cell type sample)
max_per_cell_type = int(n_cell_type * 0.03)

# Initialize list to hold samples from each cell type
cell_samples = []

# Sample from each cell type, capping at the max per type
for cell_type in cell_type_counts.index:
    cell_subset = filtered[filtered['cell_type'] == cell_type]

    # Determine how many to sample (capped at our max)
    n_to_sample = min(len(cell_subset), max_per_cell_type)

    if n_to_sample > 0:
        sample = cell_subset.sample(n=n_to_sample, random_state=42)
        cell_samples.append(sample)

# Combine all the cell type samples
cell_type_sample = pd.concat(cell_samples, ignore_index=True)

# If we have more than needed, take a random subsample
if len(cell_type_sample) > n_cell_type:
    cell_type_sample = cell_type_sample.sample(n=n_cell_type, random_state=42)
# If we have less than needed, sample more from the general population
elif len(cell_type_sample) < n_cell_type:
    remaining = n_cell_type - len(cell_type_sample)
    remaining_cells = filtered[~filtered.index.isin(cell_type_sample.index)]
    additional = remaining_cells.sample(n=remaining, random_state=42)
    cell_type_sample = pd.concat([cell_type_sample, additional], ignore_index=True)

# Part 3: Disease representation with improved balance
# Get disease counts
disease_counts = filtered['disease'].value_counts()

# Cap COVID-19 representation
max_covid = int(n_disease * 0.05)  # Cap at 5% of disease sample
max_normal = int(n_disease * 0.70)  # Allow up to 70% normal cells
max_per_disease = int(n_disease * 0.03)  # Cap other diseases at 3% each

# Initialize list to hold disease samples
disease_samples = []

# Handle COVID-19 separately
covid_cells = filtered[filtered['disease'] == 'COVID-19']
if len(covid_cells) > 0:
    n_covid = min(len(covid_cells), max_covid)
    covid_sample = covid_cells.sample(n=n_covid, random_state=42)
    disease_samples.append(covid_sample)

# Handle normal cells separately
normal_cells = filtered[filtered['disease'] == 'normal']
if len(normal_cells) > 0:
    n_normal = min(len(normal_cells), max_normal)
    normal_sample = normal_cells.sample(n=n_normal, random_state=42)
    disease_samples.append(normal_sample)

# Sample from each other disease, capping at the max per disease
for disease in disease_counts.index:
    if disease not in ['COVID-19', 'normal', 'unknown', np.nan]:
        disease_subset = filtered[filtered['disease'] == disease]

        # Determine how many to sample (capped at our max)
        n_to_sample = min(len(disease_subset), max_per_disease)

        if n_to_sample > 0:
            sample = disease_subset.sample(n=n_to_sample, random_state=42)
            disease_samples.append(sample)

# Combine all the disease samples
disease_sample = pd.concat(disease_samples, ignore_index=True)

# If we have more than needed, take a random subsample
if len(disease_sample) > n_disease:
    disease_sample = disease_sample.sample(n=n_disease, random_state=42)
# If we have less than needed, sample more from the general population
elif len(disease_sample) < n_disease:
    remaining = n_disease - len(disease_sample)
    remaining_cells = filtered[~filtered.index.isin(disease_sample.index)]
    additional = remaining_cells.sample(n=remaining, random_state=42)
    disease_sample = pd.concat([disease_sample, additional], ignore_index=True)

# Part 4: Rare tissue and developmental stage representation
# Define rare tissues (bottom 10% by frequency)
tissue_counts = filtered['tissue_general'].value_counts()
rare_threshold = tissue_counts.quantile(0.1)
rare_tissues = tissue_counts[tissue_counts <= rare_threshold].index

# Define rare developmental stages (bottom 10% by frequency)
dev_counts = filtered['dev_stage_group'].value_counts()
rare_dev_threshold = dev_counts.quantile(0.1)
rare_dev_stages = dev_counts[dev_counts <= rare_dev_threshold].index

# Get cells from rare tissues or rare developmental stages
rare_tissue_cells = filtered[filtered['tissue_general'].isin(rare_tissues)]
rare_dev_cells = filtered[filtered['dev_stage_group'].isin(rare_dev_stages)]

# Combine rare tissue and rare developmental stage cells
rare_cells = pd.concat([rare_tissue_cells, rare_dev_cells]).drop_duplicates()

if len(rare_cells) > 0:
    n_actual_rare = min(len(rare_cells), n_rare)
    rare_sample = rare_cells.sample(n=n_actual_rare, random_state=42)
else:
    # If we don't have enough rare cells, take from the general population
    rare_sample = filtered.sample(n=n_rare, random_state=42)

# Add is_disease column for later analysis
filtered['is_disease'] = ~filtered['disease'].isin(['normal', 'unknown', np.nan])

# Combine all samples
final_samples = [distribution_sample, cell_type_sample, disease_sample, rare_sample]
final_sample = pd.concat(final_samples, ignore_index=True)

# Add is_disease column for analysis
final_sample['is_disease'] = ~final_sample['disease'].isin(['normal', 'unknown', np.nan])

# Remove potential duplicates
final_sample = final_sample.drop_duplicates()

# Adjust to target size
if len(final_sample) > target_total:
    final_sample = final_sample.sample(n=target_total, random_state=42)
elif len(final_sample) < target_total:
    remaining = target_total - len(final_sample)
    remaining_cells = filtered[~filtered.index.isin(final_sample.index)]
    if len(remaining_cells) >= remaining:
        additional = remaining_cells.sample(n=remaining, random_state=42)
        final_sample = pd.concat([final_sample, additional], ignore_index=True)
    else:
        additional = filtered.sample(n=remaining, replace=True, random_state=42)
        final_sample = pd.concat([final_sample, additional], ignore_index=True)

# Final analysis of our sample
print(f"Final dataset size: {len(final_sample)} cells")

# Compare distributions of key variables
variables = ['tissue_general', 'dev_stage_group', 'cell_type', 'sex', 'is_disease', 'disease']

for var in variables:
    if var in filtered.columns:
        orig_dist = filtered[var].value_counts(normalize=True).head(10)
        sample_dist = final_sample[var].value_counts(normalize=True).head(10)

        print(f"\nTop 10 {var} distribution (original):")
        print(orig_dist)

        print(f"\nTop 10 {var} distribution (sampled):")
        print(sample_dist)

        # Calculate percentage change for key categories
        if var in ['tissue_general']:
            print("\nKey tissue changes (original → sampled):")
            for tissue in ['brain', 'blood', 'lung', 'eye', 'breast']:
                orig_pct = orig_dist.get(tissue, 0) * 100
                sample_pct = sample_dist.get(tissue, 0) * 100
                change = sample_pct - orig_pct
                print(f"{tissue}: {orig_pct:.1f}% → {sample_pct:.1f}% ({change:+.1f}%)")

        if var == 'disease':
            print("\nKey disease changes (original → sampled):")
            for disease in ['normal', 'COVID-19', 'Parkinson disease', 'lung adenocarcinoma']:
                if disease in filtered['disease'].values:
                    orig_count = filtered[filtered['disease'] == disease].shape[0]
                    orig_pct = (orig_count / len(filtered)) * 100

                    sample_count = final_sample[final_sample['disease'] == disease].shape[0]
                    sample_pct = (sample_count / len(final_sample)) * 100

                    change = sample_pct - orig_pct
                    print(f"{disease}: {orig_pct:.1f}% → {sample_pct:.1f}% ({change:+.1f}%)")

# Check disease representation
disease_orig_pct = filtered[filtered['is_disease']].shape[0] / len(filtered) * 100
disease_sample_pct = final_sample[final_sample['is_disease']].shape[0] / len(final_sample) * 100
change = disease_sample_pct - disease_orig_pct

print(f"\nDisease samples: {disease_orig_pct:.1f}% → {disease_sample_pct:.1f}% ({change:+.1f}%)")

# Check rare category representation
rare_tissue_orig_pct = filtered[filtered['tissue_general'].isin(rare_tissues)].shape[0] / len(filtered) * 100
rare_tissue_sample_pct = final_sample[final_sample['tissue_general'].isin(rare_tissues)].shape[0] / len(final_sample) * 100

rare_dev_orig_pct = filtered[filtered['dev_stage_group'].isin(rare_dev_stages)].shape[0] / len(filtered) * 100
rare_dev_sample_pct = final_sample[final_sample['dev_stage_group'].isin(rare_dev_stages)].shape[0] / len(final_sample) * 100

print(f"\nRare tissues: {rare_tissue_orig_pct:.2f}% → {rare_tissue_sample_pct:.2f}% ({rare_tissue_sample_pct - rare_tissue_orig_pct:+.2f}%)")
print(f"Rare dev stages: {rare_dev_orig_pct:.2f}% → {rare_dev_sample_pct:.2f}% ({rare_dev_sample_pct - rare_dev_orig_pct:+.2f}%)")

# Get cell type distribution for further analysis
print("\nDetailed cell type distribution in sampled dataset:")
cell_type_dist = final_sample['cell_type'].value_counts().head(15)
print(cell_type_dist)
print(f"Number of unique cell types in sample: {final_sample['cell_type'].nunique()}")

# Get disease distribution for further analysis
print("\nDetailed disease distribution in sampled dataset:")
disease_dist = final_sample['disease'].value_counts().head(15)
print(disease_dist)
print(f"Number of unique diseases in sample: {final_sample['disease'].nunique()}")

# Save the final sampled dataset
# final_sample.to_csv('balanced_cell_sample.csv', index=False)

Final dataset size: 100000 cells

Top 10 tissue_general distribution (original):
tissue_general
brain              0.333059
blood              0.185784
lung               0.067297
eye                0.063202
heart              0.045586
breast             0.043336
liver              0.023374
kidney             0.020807
small intestine    0.019634
skin of body       0.013879
Name: proportion, dtype: float64

Top 10 tissue_general distribution (sampled):
tissue_general
brain              0.19111
blood              0.14817
lung               0.08558
eye                0.06184
heart              0.04408
breast             0.04000
kidney             0.03995
endocrine gland    0.03958
small intestine    0.03403
liver              0.02757
Name: proportion, dtype: float64

Key tissue changes (original → sampled):
brain: 33.3% → 19.1% (-14.2%)
blood: 18.6% → 14.8% (-3.8%)
lung: 6.7% → 8.6% (+1.8%)
eye: 6.3% → 6.2% (-0.1%)
breast: 4.3% → 4.0% (-0.3%)

Top 10 dev_stage_group distribution (original

In [ ]:

# Print summary distributions
print("\n📊 Distribution after sampling:\n")

pd.set_option("display.max_rows", None)


# Tissue_general distribution
print("🧠 Tissue (tissue_general):")
print(final_sample["tissue_general"].value_counts(), "\n")


print("🧠 Tissue (tissue):")
print(final_sample["tissue"].value_counts(), "\n")


# Development stage group
print("👶 Development Stage:")
print(final_sample["dev_stage_group"].value_counts(), "\n")

# Cell type
print("🔬 Cell Type:")
print(final_sample["cell_type"].value_counts(), "\n")

# Sex
print("⚧️ Sex:")
print(final_sample["sex"].value_counts(), "\n")

# Disease
print("🦠 Disease:")
print(final_sample["disease"].value_counts(), "\n")



print("Total cells:", len(final_sample))


📊 Distribution after sampling:

🧠 Tissue (tissue_general):
tissue_general
brain                        19111
blood                        14817
lung                          8558
eye                           6184
heart                         4408
breast                        4000
kidney                        3995
endocrine gland               3958
small intestine               3403
liver                         2757
skin of body                  2654
bone marrow                   2515
colon                         2003
respiratory system            1718
placenta                      1417
pancreas                      1100
musculature                   1077
nose                           987
lymph node                     976
digestive system               848
stomach                        754
exocrine gland                 748
adipose tissue                 747
intestine                      696
prostate gland                 660
reproductive system            656
immune system  

In [ ]:
join_ids = final_sample["soma_joinid"].tolist()
sorted_join_ids = sorted(final_sample['soma_joinid'].unique())



In [ ]:
# Clear variables from memory

for name in list(globals()):
    if name in ('filtered', 'final_sample', 'remaining_cells', 'normal_cells', 'weights', 'covid_cells', 'rare_cells', 'join_ids') :
        globals().pop(name)

# Force garbage collection
gc.collect()



0

In [ ]:

# Define batch size
batch_size = 2000
all_data = []

# Process in batches using ID ranges
for i in range(0, len(sorted_join_ids), batch_size):
    batch_ids = sorted_join_ids[i:i+batch_size]
    join_ids_str = ",".join(map(str, batch_ids))
    if not batch_ids:
        continue


    print(f"Processing batch {i//batch_size + 1}")

    # Query using range instead of membership
    batch_data = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens",
        obs_value_filter=f"soma_joinid  in [{join_ids_str}]",
        obs_column_names=["soma_joinid", "sex", "tissue", "tissue_ontology_term_id", "tissue_general", 'tissue_general_ontology_term_id',
                          "cell_type", "cell_type_ontology_term_id", "disease_ontology_term_id",
                          "disease", "development_stage"],
    )

    # these are necesary for the tokenization with Geneformer
    if 'ensembl_id' not in batch_data.var.columns:
        batch_data.var['ensembl_id'] = batch_data.var['feature_id']


    if scipy.sparse.issparse(batch_data.X):
        batch_data.obs["n_counts"] = np.array(batch_data.X.sum(axis=1)).flatten()
    else:
        batch_data.obs["n_counts"] = batch_data.X.sum(axis=1)

    # CELLxGENE return the raw values at adata.X and Geneforemr expect the raw values at adata.raw.X
    if batch_data.raw is None:
      batch_data.raw = batch_data.copy()

    # Save results
    batch_data.write(f"/content/drive/MyDrive/cell2text_dataset/batch_{i}.h5ad")


    del batch_data
    gc.collect()



Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Processing batch 8
Processing batch 9
Processing batch 10
Processing batch 11
Processing batch 12
Processing batch 13
Processing batch 14
Processing batch 15
Processing batch 16
Processing batch 17
Processing batch 18
Processing batch 19
Processing batch 20
Processing batch 21
Processing batch 22
Processing batch 23
Processing batch 24
Processing batch 25
Processing batch 26
Processing batch 27
Processing batch 28
Processing batch 29
Processing batch 30
Processing batch 31
Processing batch 32
Processing batch 33
Processing batch 34
Processing batch 35
Processing batch 36
Processing batch 37
Processing batch 38
Processing batch 39
Processing batch 40
Processing batch 41
Processing batch 42
Processing batch 43
Processing batch 44
Processing batch 45
Processing batch 46
Processing batch 47
Processing batch 48
Processing batch 49
Processing batch 50
